# Getting Started with LangChain — Updated LCEL Notebook

This notebook demonstrates the same introductory LangChain concepts as the original `01-intro.ipynb`, but with:

- different source data,
- current `langchain_core` imports,
- LCEL-style chains,
- `StrOutputParser`,
- Pydantic structured output,
- and an end-to-end multi-step chain.

The example task: turn rough engineering/course notes into useful teaching material.

## 1. Install dependencies

Run this cell in a fresh environment.  
For production notebooks, pin versions in `requirements.txt`. For a teaching notebook, using `-U` keeps the API current.

In [1]:
%pip install anthropic

   ---------------------------------------- 0.0/831.7 kB ? eta -:--:--
   ------------ --------------------------- 262.1/831.7 kB ? eta -:--:--
   ---------------------------------------- 831.7/831.7 kB 5.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
import anthropic
import os

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

try:
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": "list all available anthropic models."}],
    )
    print("✅ API key works! Response:", message.content[0].text)
except anthropic.AuthenticationError:
    print("❌ Invalid API key.")
except Exception as e:
    print(f"❌ Error: {e}")

C:\Users\User\AppData\Local\Temp\ipykernel_4268\2253566786.py:7: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(


✅ API key works! Response: Here are the currently available Anthropic models:

## Claude 3.5 Series
- **Claude 3.5 Sonnet** - The most advanced model, excellent for complex reasoning, coding, and creative tasks
- **Claude 3.5 Haiku** - Fast and efficient model for simpler tasks

## Claude 3 Series
- **Claude 3 Opus** - The most capable Claude 3 model for complex tasks
- **Claude 3 Sonnet** - Balanced performance and speed
- **Claude 3 Haiku** - Fastest and most cost-effective Claude 3 model

## Legacy Models
- **Claude 2.1** - Previous generation model
- **Claude 2.0** - Earlier version
- **Claude Instant 1.2** - Faster, lighter-weight model

## Key Notes:
- **Claude 3.5 Sonnet** is currently the flagship model with the best overall performance
- Models vary in capabilities, speed, and cost
- Availability may depend on your access tier (free vs. paid plans)
- Some models may have different context windows and rate limits

The exact model names and availability can change, so I'd recomm

In [3]:
# If needed, uncomment and run:
%pip install -U langchain langchain-core langchain-openai pydantic

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 10.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/876.5 kB ? eta -:--:--
   --------------------------------------- 876.5/876.5 kB 38.4 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.5
    Uninstalling pydantic-2.12.5:
      Successfully uninstalled pydantic-2.12.5
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Configure the OpenAI API key

`ChatOpenAI` reads `OPENAI_API_KEY` from the environment.

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Initialize chat models

We define two model objects:

- `llm`: deterministic output for stable tasks.
- `creative_llm`: higher temperature for naming and creative writing.

You may replace the model name with another chat model supported by your account.

In [9]:
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4.1-mini"

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
creative_llm = ChatOpenAI(model=MODEL_NAME, temperature=0.8)

## 4. Define different source data

Instead of the article used in the source notebook, we use rough notes for a short engineering lecture.

The data combines:
- a raw technical note,
- target audience details,
- and desired learning outcomes.

In [10]:
technical_note = """
A manufacturing plant collects vibration, temperature, and acoustic readings from rotating machines.
The maintenance team wants to detect early signs of bearing faults before a machine fails.

Today, most alerts are based on fixed thresholds. This is simple, but it creates many false alarms
and misses weak signals that appear only as combinations of features.

A better approach is to build a predictive maintenance workflow:
1. collect time-windowed sensor features,
2. label historical failure events,
3. train a supervised classifier or anomaly detector,
4. evaluate the model using precision, recall, and alert lead time,
5. deploy the model with monitoring, human review, and periodic retraining.

The goal is not to replace engineers. The goal is to prioritize inspections and reduce unplanned downtime.
"""

audience_profile = {
    "audience": "mechanical and industrial engineers",
    "level": "introductory AI course",
    "duration": "45 minutes",
    "tone": "practical, clear, not mathematical-heavy",
}

learning_goals = [
    "Explain predictive maintenance as an ML use case",
    "Distinguish threshold rules from learned models",
    "Identify suitable input features and labels",
    "Connect model metrics to maintenance decisions",
]

## 5. Build a prompt template

Current LangChain code should usually import prompt classes from `langchain_core.prompts`.

A `ChatPromptTemplate` represents a list of chat messages with variables.

In [11]:
from langchain_core.prompts import ChatPromptTemplate

title_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You help create concise titles for engineering AI teaching material."
    ),
    (
        "human",
        """
Create one clear and engaging lecture title.

Technical note:
{technical_note}

Audience:
{audience}

Level:
{level}

Rules:
- Output only the title.
- Do not use quotation marks.
- Maximum 12 words.
"""
    )
])

formatted_messages = title_prompt.invoke({
    "technical_note": "TEST NOTE",
    "audience": "engineers",
    "level": "introductory",
})

formatted_messages

ChatPromptValue(messages=[SystemMessage(content='You help create concise titles for engineering AI teaching material.', additional_kwargs={}, response_metadata={}), HumanMessage(content='\nCreate one clear and engaging lecture title.\n\nTechnical note:\nTEST NOTE\n\nAudience:\nengineers\n\nLevel:\nintroductory\n\nRules:\n- Output only the title.\n- Do not use quotation marks.\n- Maximum 12 words.\n', additional_kwargs={}, response_metadata={})])

## 6. Create a simple LCEL chain

LCEL uses the pipe operator `|`.

The flow here is:

`input dict -> prompt -> chat model -> string parser`

In [12]:
from langchain_core.output_parsers import StrOutputParser

title_chain = title_prompt | creative_llm | StrOutputParser()

title = title_chain.invoke({
    "technical_note": technical_note,
    "audience": audience_profile["audience"],
    "level": audience_profile["level"],
})

title

'Predictive Maintenance with AI for Early Bearing Fault Detection'

## 7. Build a second chain that depends on earlier output

The next prompt receives both the original technical note and the generated title.

In [ ]:
abstract_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You write compact course descriptions for technical learners."
    ),
    (
        "human",
        """
Write a short lecture abstract.

Title:
{title}

Technical note:
{technical_note}

Audience profile:
{audience_profile}

Learning goals:
{learning_goals}

Rules:
- 80-120 words.
- Mention the practical engineering value.
- Avoid hype.
"""
    )
])

abstract_chain = abstract_prompt | llm | StrOutputParser()

abstract = abstract_chain.invoke({
    "title": title,
    "technical_note": technical_note,
    "audience_profile": audience_profile,
    "learning_goals": learning_goals,
})

print(abstract)

## 8. Structured output with Pydantic

Free text is useful for humans, but applications often need structured objects.

Here we ask the model to review the lecture material and return a validated Pydantic object.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class LectureReview(BaseModel):
    strengths: List[str] = Field(description="Strong parts of the lecture material")
    risks: List[str] = Field(description="Possible weaknesses, gaps, or confusing parts")
    suggested_example: str = Field(description="A concrete example to add")
    revised_learning_goals: List[str] = Field(description="Improved learning goals")

review_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert reviewer of AI course material for engineers."
    ),
    (
        "human",
        """
Review this lecture material.

Title:
{title}

Abstract:
{abstract}

Technical note:
{technical_note}

Original learning goals:
{learning_goals}

Return a structured review.
"""
    )
])

structured_review_llm = llm.with_structured_output(LectureReview)

review_chain = review_prompt | structured_review_llm

review = review_chain.invoke({
    "title": title,
    "abstract": abstract,
    "technical_note": technical_note,
    "learning_goals": learning_goals,
})

review

The result is a Python object, not a string.  
That makes it easier to use downstream.

In [ ]:
print("Strengths:")
for item in review.strengths:
    print("-", item)

print("\nRisks:")
for item in review.risks:
    print("-", item)

print("\nSuggested example:")
print(review.suggested_example)

## 9. Use `RunnablePassthrough.assign` to build a multi-step workflow

`RunnablePassthrough.assign(...)` lets us add new fields to the input dictionary while keeping the original fields.

This is useful when later steps need both:
- the original inputs,
- and outputs from earlier model calls.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

pipeline = (
    RunnablePassthrough.assign(
        title=title_chain
    )
    | RunnablePassthrough.assign(
        abstract=abstract_chain
    )
    | RunnablePassthrough.assign(
        review=review_chain
    )
)

pipeline_result = pipeline.invoke({
    "technical_note": technical_note,
    "audience": audience_profile["audience"],
    "level": audience_profile["level"],
    "audience_profile": audience_profile,
    "learning_goals": learning_goals,
})

pipeline_result.keys()

In [ ]:
print("TITLE")
print(pipeline_result["title"])

print("\nABSTRACT")
print(pipeline_result["abstract"])

print("\nREVIEW OBJECT")
pipeline_result["review"]

## 10. Add one more generation step: slide outline

Now we use the accumulated fields to produce a short slide outline.

In [ ]:
slide_outline_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You design concise slide outlines for technical lectures."
    ),
    (
        "human",
        """
Create a 6-slide outline.

Title:
{title}

Abstract:
{abstract}

Review:
{review}

Technical note:
{technical_note}

Rules:
- Use exactly 6 slides.
- Each slide should have a title and 2-3 bullets.
- Keep it practical for engineers.
"""
    )
])

slide_outline_chain = slide_outline_prompt | llm | StrOutputParser()

outline = slide_outline_chain.invoke(pipeline_result)

print(outline)

## 11. Optional: create a visual brief instead of generating an image

The original notebook includes a visual-generation style step.  
Here we keep the notebook API-light and generate a prompt/brief that could later be sent to an image model.

In [ ]:
visual_brief_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You create clear visual briefs for educational slide thumbnails."
    ),
    (
        "human",
        """
Create a visual brief for a lecture thumbnail.

Lecture title:
{title}

Lecture abstract:
{abstract}

Rules:
- No text inside the image.
- Describe a clean visual metaphor.
- Mention composition, objects, and style.
- Keep it under 80 words.
"""
    )
])

visual_brief_chain = visual_brief_prompt | creative_llm | StrOutputParser()

visual_brief = visual_brief_chain.invoke(pipeline_result)
print(visual_brief)

## 12. Summary

This notebook demonstrated:

1. `ChatOpenAI` initialization.
2. `ChatPromptTemplate` for system/human messages.
3. LCEL chaining with `|`.
4. `StrOutputParser` for plain text outputs.
5. Pydantic structured output with `.with_structured_output(...)`.
6. `RunnablePassthrough.assign(...)` for multi-step workflows.

The important design idea is to pass structured intermediate values forward instead of manually copying strings between cells.